# v4 article extraction — scored against the v4 gold article list

The frozen v4 pipeline (tag `extraction-v4`) on the **held-out v4 test volumes** (Penny Magazine 1843, Saturday
Magazine 1843, Chambers's Edinburgh Journal 1837 — three issues each, never used to write rules) scored against
`gold_article_list_test_v4.csv`, which Owen annotated from the printed pages alone, without seeing the output
(guide: `documentation/article_extraction_pipeline_iterative_build/gold_article_list_annotation_guide.md`).
The gold list was corrected afterwards from the printed pages only: Penny issues 692–693 dated 1843 (entered as
1834), Chambers No. 272 dated April 15, 1837 (entered as 4.16), and Chambers gold 81 opening "The actions of the"
(entered as "The sections of the"). The output read is `output/article_extraction/v4/`, frozen. Page numbers are
PDF pages; the check images are in `output/article_extraction/v4/<volume>/reading_order/`. What v4 changed is in
`documentation/article_extraction_pipeline_iterative_build/v4_decisions.md`; the fixes made after this test are
in `v5_decisions.md`.

**How items are matched.** Each gold item is located in the output by its first and last words
(OCR-tolerant matching on the item's own pages). When the same opening words also appear earlier, quoted inside
another article, an equally good match at an article's start wins (added for this test; it changed no v2 or v3
score). Every number below is computed by `score_gold.py`; nothing is judged by hand.

**The v4 gold list** splits the old `notes` column into `figure_notes` (the "fig:" entries counted for
illustrations) and `item_notes` (Owen's description of each item: titles, sign-offs, printed lines, footnotes).
Section 10 sets the item notes beside each error.

| Outcome | Meaning |
|---|---|
| **exact** | the item opens a pipeline article of its own, and its last words are in that article |
| **merged** | its pipeline article also holds the start of the gold item before or after it (a missed boundary) |
| **split** | its first and last words land in different pipeline articles (a false boundary inside it) |
| **lost** | its first words are in no article's text (set aside as furniture, absorbed into other matter, or not found) |


In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

HERE = next(p for p in [Path("."), Path("evaluation/evaluating_v4_article_extraction")]
            if (p / "gold_article_list_test_v4.csv").exists())
SCORER_DIR = HERE.resolve().parent / "evaluating_v2_article_extraction"  # score_gold.py lives there
sys.path.insert(0, str(SCORER_DIR))
from score_gold import load_gold, load_pipeline, score, summary_tables, write, VOLUME_ALIASES

VERSION = "v4"
GOLD = HERE / "gold_article_list_test_v4.csv"
OUT_DIR = HERE  # results: <VERSION>_gold_comp_eval_*.csv / _summary.md

## Run the scoring and save the results

In [2]:
gold = load_gold(GOLD)
res = score(VERSION, gold)
summary_path = write(res, VERSION, OUT_DIR, GOLD)
items, arts, pieces, imprints = res["items"], res["pipeline_articles"], res["pieces"], res["imprints"]
tables = summary_tables(res)
own = items[items.opens_own_article]
print(f"Gold list: {len(gold)} rows — {gold.article_type.value_counts().to_dict()}")
print(f"Scored {len(items)} gold items against {len(arts)} pipeline articles ({VERSION})")
print(f"Wrote {OUT_DIR}/{VERSION}_gold_comp_eval_*.csv and {summary_path.name}")

Gold list: 124 rows — {'article': 117, 'imprint': 6, 'editorial_notice': 1}
Scored 118 gold items against 122 pipeline articles (v4)
Wrote ./v4_gold_comp_eval_*.csv and v4_gold_comp_eval_summary.md


## 1. Headline: items and boundaries

In [3]:
tables['items']

,penny_1843,saturday_1843,chambers_1837,all
gold items,20,26,72,118
pipeline articles,18,27,77,122
exact,16/20 (80%),17/26 (65%),62/72 (86%),95/118 (81%)
merged,4/20 (20%),6/26 (23%),4/72 (6%),14/118 (12%)
split,0/20 (0%),0/26 (0%),5/72 (7%),5/118 (4%)
lost,0/20 (0%),3/26 (12%),1/72 (1%),4/118 (3%)
boundary recall,18/20 (90%),20/26 (77%),69/72 (96%),107/118 (91%)
boundary precision,18/18 (100%),20/27 (74%),69/77 (90%),107/122 (88%)
extra pipeline articles,0,7,8,15


In [4]:
# Item outcome counts per volume
pd.crosstab(items.volume, items.status, margins=True, margins_name="total")

status,exact,lost,merged,split,total
volume,,,,,
chambers_1837,62,1,4,5,72
penny_1843,16,0,4,0,20
saturday_1843,17,3,6,0,26
total,95,4,14,5,118


## 2. Successes — items extracted exactly

In [5]:
exact = items[items.status == "exact"]
print(f"{len(exact)} of {len(items)} gold items extracted exactly")
exact[["volume", "gold_id", "gold_title", "pipeline_title", "start_article_id"]]

95 of 118 gold items extracted exactly


,volume,gold_id,gold_title,pipeline_title,start_article_id
0,penny_1843,1,Sir Roger De Coverley.—No. II.,SIR ROGER DE COVERLEY.-No. I,penny_1843_0001
1,penny_1843,2,Portable Diorama.—Dissolving Views.,PORTABLE DIORAMA. - DISSOLVING VIEWS,penny_1843_0002
2,penny_1843,3,Bats.,BATS,penny_1843_0003
3,penny_1843,4,The Privileges and Liabilities of British Shipping.,THE PRIVILEGES AND LIABILITIES OF BRITISH SHIPPING,penny_1843_0004
4,penny_1843,5,Usefulness of Moles.,Usefulness of Moles,penny_1843_0005
5,penny_1843,6,Autumnal Traveling in Siberia.,Autumnal Travelling in Siberia,penny_1843_0006
6,penny_1843,7,Importance of Cities.,Importance of Cities,penny_1843_0007
7,penny_1843,8,Fossil Trees.,Fossil Trees,penny_1843_0008
8,penny_1843,9,The Castes and Tribes of India.,,penny_1843_0009
13,penny_1843,14,The Arein.,The Arein,penny_1843_0012


## 3. Errors — merged items (missed boundaries)

In [6]:
items[items.status.str.startswith("merged")][
    ["volume", "gold_id", "start_pdf_page", "gold_title", "merged_with", "start_article_id", "pipeline_title"]]

,volume,gold_id,start_pdf_page,gold_title,merged_with,start_article_id,pipeline_title
9,penny_1843,10,15,Walhalla,11,penny_1843_0010,THE WALHALLA
10,penny_1843,11,16,Pekin.,10,penny_1843_0010,
11,penny_1843,12,17,Progresses of Queen Elizabeth. From Hatfield to her Coronation.,13,penny_1843_0011,PROGRESSES OF QUEEN ELIZAВЕТН
12,penny_1843,13,20,Insurance of Shipping.—“Lloyd’s List.”,12,penny_1843_0011,
29,saturday_1843,30,11,,31,saturday_1843_0010,
30,saturday_1843,31,11,,30,saturday_1843_0010,
33,saturday_1843,35,13,Holyrood House.,36,saturday_1843_0013,HOLYROOD HOUSE
34,saturday_1843,36,15,,35,saturday_1843_0013,
39,saturday_1843,41,17,The Application of Electricity to the Blasting of Rocks.,42,saturday_1843_0018,THE APPLICATION OF ELECTRICITY TO THE BLASTING OF ROCKS
40,saturday_1843,42,19,,41,saturday_1843_0018,


## 4. Errors — split items (false boundaries inside an item)

In [7]:
split = items[items.status.str.contains("split")]
split[["volume", "gold_id", "start_pdf_page", "gold_title", "start_article_id", "end_article_id", "pipeline_articles"]]

,volume,gold_id,start_pdf_page,gold_title,start_article_id,end_article_id,pipeline_articles
50,chambers_1837,54,11,Yankee Humour.,chambers_1837_0005,chambers_1837_0007,chambers_1837_0005;chambers_1837_0006;chambers_1837_0007
71,chambers_1837,76,19,Thoughts on Commonplace Subjects,chambers_1837_0028,chambers_1837_0029,chambers_1837_0028;chambers_1837_0029
75,chambers_1837,80,22,Present Condition of Belgium.,chambers_1837_0033,chambers_1837_0035,chambers_1837_0033;chambers_1837_0034;chambers_1837_0035
105,chambers_1837,111,29,Loss of British Shipping,chambers_1837_0065,chambers_1837_0066,chambers_1837_0065;chambers_1837_0066
106,chambers_1837,112,29,Poetry of Many Lands.,chambers_1837_0067,chambers_1837_0068,chambers_1837_0067;chambers_1837_0068


## 5. Errors — lost items (opening words not in any article's text)

In [8]:
items[items.status == "lost"][["volume", "gold_id", "start_pdf_page", "gold_title", "start_found_in", "start_score"]]

,volume,gold_id,start_pdf_page,gold_title,start_found_in,start_score
21,saturday_1843,22,7,,front_matter,1.00
43,saturday_1843,45,19,Chess,no gold words,0.00
45,saturday_1843,48,28,,box role: furniture,1.00
107,chambers_1837,113,30,Management and Mismanagement.,back_matter,0.75


## 6. Extra pipeline articles (hold no gold item's start)

Usually fragments of split items (`gold_ends` shows a gold item whose *last* words fall in it) or page
furniture taken as an article (masthead lines, prices, dates).

In [9]:
extra = arts[arts.status == "extra"]
print(extra.volume.value_counts().to_dict())
extra[["volume", "article_id", "pipeline_title", "pages", "word_count", "gold_ends", "text_start"]]

{'chambers_1837': 8, 'saturday_1843': 7}


,volume,article_id,pipeline_title,pages,word_count,gold_ends,text_start
18,saturday_1843,saturday_1843_0001,Saturday WITHOUT SOUL BE COODL THAT THE,5,0,,
38,saturday_1843,saturday_1843_0021,CHESS,19;20,1210,,BEFORE the close of the year it may be desirable to give
39,saturday_1843,saturday_1843_0022,"PROBLEM XXI., page 152",20,42,,1. Kt. takes K. P. checking. 1. K. B. takes Kt 2.
40,saturday_1843,saturday_1843_0023,"PROBLEM XXII., page 224",20,43,,1. Q. to Q. sixth square. 1. K. to K. fifth square.
41,saturday_1843,saturday_1843_0024,SOLUTIONS TO CURIOUS CHESS PROBLEMS,20,581,,WHITE. BLACK. 1. Q. to K. Kt. second square.. 1. R. takes
42,saturday_1843,saturday_1843_0025,BLACK,20,106,,BLACK. WHITE. 1. K. to K. R. third square. 1. R. to
43,saturday_1843,saturday_1843_0026,WHITE,20,31,,WHITE. BLACK 1. Kt. to K. Kt. sixth sq. chg. 1. Kt.
50,chambers_1837,chambers_1837_0006,SOFT SAWDER AND HUMAN NATUR,11,1116,,In the course of a journey which Mr Slick performs in company
51,chambers_1837,chambers_1837_0007,NOVA-SCOTIAN HORSES,11,672,54,Among the absurd practices charged by the Clockmaker against the N...
52,chambers_1837,chambers_1837_0008,ILLUSTRATION OF SUBTERRANEAN WEALTH,12,612,,ILLUSTRATION OF SUBTERRANEAN WEALTH. I tell good above and better ...


## 7. Fields — summary

In [10]:
tables['fields']

,penny_1843,saturday_1843,chambers_1837,all
items scored (open their own article),18,20,69,107
title,16/18 (89%),18/20 (90%),68/69 (99%),102/107 (95%)
front matter pieces in place,–,1/2 (50%),2/5 (40%),3/7 (43%)
back matter pieces in place,6/9 (67%),5/7 (71%),20/27 (74%),31/43 (72%)
series (where either has one),1/1 (100%),0/3 (0%),0/2 (0%),1/6 (17%)
continued from,17/18 (94%),18/20 (90%),67/69 (97%),102/107 (95%)
to be continued,18/18 (100%),17/20 (85%),69/69 (100%),104/107 (97%)
editorial notice / article,18/18 (100%),20/20 (100%),69/69 (100%),107/107 (100%)
illustration count,16/18 (89%),16/20 (80%),69/69 (100%),101/107 (94%)
issue number,0/18 (0%),20/20 (100%),69/69 (100%),89/107 (83%)


### 7a. Titles that disagree

In [11]:
own[~own.title_result.isin(["correct", "both untitled"])][
    ["volume", "gold_id", "start_pdf_page", "gold_title", "pipeline_title", "title_result"]]

,volume,gold_id,start_pdf_page,gold_title,pipeline_title,title_result
8,penny_1843,9,14,The Castes and Tribes of India.,,missing
11,penny_1843,12,17,Progresses of Queen Elizabeth. From Hatfield to her Coronation.,PROGRESSES OF QUEEN ELIZAВЕТН,wrong
22,saturday_1843,23,7,,COLOSSAL STATUE DRAWN TO ITS PLACE,extra (gold untitled)
44,saturday_1843,47,22,The History of Copper-Plate Engraving.—Concluded,SECTION 1. EARLY BRITISH ENGRAVING,wrong
106,chambers_1837,112,29,Poetry of Many Lands.,FRANCE. BERANGER,wrong


### 7b. Front and back matter pieces

Each gold piece is found **in place** (the pipeline's front/back matter), **in text** (left in the article text),
elsewhere, or **missing** (set aside or not found). Footnotes — many in Chambers — are counted separately; some
give sources, others explain a word or concept.

In [12]:
tables['pieces']

result,in place,in text,in title,missing,total
kind,,,,,
back matter,25,9,0,0,34
back matter (footnote),6,2,0,1,9
front matter,3,2,2,0,7
total,34,13,2,1,50


In [13]:
# Pieces not in place
pieces[pieces.result != "in place"][["volume", "gold_id", "field", "footnote", "result", "piece"]]

,volume,gold_id,field,footnote,result,piece
0,penny_1843,6,back_matter,False,in text,Cottrell’s Recollections of Siberia.
3,penny_1843,10,back_matter,True,in text,* How few and comparatively small apertures are required for light...
4,penny_1843,12,back_matter,True,in text,"* A noise is a band - a noise of music, a band of music."
11,saturday_1843,28,back_matter,False,in text,Milner.
14,saturday_1843,38,back_matter,False,in text,Guizot.
17,saturday_1843,47,front_matter,False,in title,Section 1. Early British Engraving.
24,chambers_1837,64,front_matter,False,in text,Translated from the Italian of Redi
26,chambers_1837,66,back_matter,False,in text,Newspaper paragraph.
29,chambers_1837,73,front_matter,False,in title,By Miss Mitford.
31,chambers_1837,81,back_matter,False,in text,"From a forthcoming work by Dr Busham of Dumfries, entitled “The Ph..."


In [14]:
# Pipeline front/back matter with no gold counterpart (extra pieces), per item
x = own[(own.front_matter_extra > 0) | (own.back_matter_extra > 0)]
x[["volume", "gold_id", "gold_title", "front_matter_extra", "front_matter_pipeline",
   "back_matter_extra", "back_matter_pipeline"]]

,volume,gold_id,gold_title,front_matter_extra,front_matter_pipeline,back_matter_extra,back_matter_pipeline
22,saturday_1843,23,,1.0,9110 A GOOD man submits his mind to the Governor of the whole univ...,0.0,EPICTETUS. 2200ml
33,saturday_1843,35,Holyrood House.,0.0,"Then list, ye maidens, to my lay, Though old the tale, and past th...",1.0,Encyclopedia
52,chambers_1837,56,My Friend Bromely.,0.0,,1.0,"*I did get him out of bed, and his clothes on. He fainted once, an..."
71,chambers_1837,76,Thoughts on Commonplace Subjects,1.0,ENGLISH AND FRENCH.,0.0,
101,chambers_1837,107,The Bragge Family.,0.0,,1.0,[In the preceding paper the reader will probably recognise the liv...
102,chambers_1837,108,Betting at Newmarket and Doncaster.,0.0,,1.0,"* Education Reform, by Thomas Wyse, Esq. M.P."
105,chambers_1837,111,Loss of British Shipping,0.0,,1.0,"Tait's Magazine, April 1837"
106,chambers_1837,112,Poetry of Many Lands.,0.0,,2.0,"MANAGEMENT AND MISMANAGEMENT.* GREAT men have written their ""confe..."


### 7c. Series and continuations that disagree

In [15]:
g = gold.set_index(["volume", "gold_id"])
s = own[~own.series_result.isin(["correct", "none (both)"])].copy()
s["gold_series"] = [g.loc[(v, i), "series"] for v, i in zip(s.volume, s.gold_id)]
s[["volume", "gold_id", "gold_title", "gold_series", "pipeline_series", "series_result"]]

,volume,gold_id,gold_title,gold_series,pipeline_series,series_result
33,saturday_1843,35,Holyrood House.,The Royal Residences of Scotland VI.,Svinadat Herutar Be Soul the Royal,wrong
38,saturday_1843,40,On Dioramic Painting. II. Portable Dioramas,,Ot Yton Dioramic Painting a Ht,extra
44,saturday_1843,47,The History of Copper-Plate Engraving.—Concluded,A Brief History of the Fine Arts. VIII.,The History of Copper Plate Engraving Concluded,wrong
53,chambers_1837,57,Traits of Spanish Misrule,,He Has Not Touched Card Nor Dice Box Since,extra
106,chambers_1837,112,Poetry of Many Lands.,,Poetry of Many Lands,extra


In [16]:
own[~own.continued_from_result.str.startswith("correct") | ~own.to_be_continued_result.str.startswith("correct")][
    ["volume", "gold_id", "gold_title", "continued_from_result", "to_be_continued_result"]]

,volume,gold_id,gold_title,continued_from_result,to_be_continued_result
17,penny_1843,18,The Trinity House.,extra,correct (no)
26,saturday_1843,27,Plant-Like Animals. VI.,missing,missing
28,saturday_1843,29,On Dioramic Painting. I.,correct (no),missing
38,saturday_1843,40,On Dioramic Painting. II. Portable Dioramas,missing,missing
73,chambers_1837,78,Texas.—Third Article.,missing,correct (no)
97,chambers_1837,102,Post-Office in France.,extra,correct (no)


### 7d. Illustrations and article type that disagree

In [17]:
own[own.figures_result != "correct"][["volume", "gold_id", "gold_title", "figures_gold", "figures_pipeline"]]

,volume,gold_id,gold_title,figures_gold,figures_pipeline
15,penny_1843,16,The Felling and transport of the Pine and Fire.,0.0,1.0
16,penny_1843,17,Etruscan Antiquities at the British Museum.,2.0,1.0
20,saturday_1843,21,Egyptian Sculpture.,2.0,1.0
26,saturday_1843,27,Plant-Like Animals. VI.,3.0,2.0
31,saturday_1843,32,On Quarrying Stone.,3.0,1.0
39,saturday_1843,41,The Application of Electricity to the Blasting of Rocks.,1.0,0.0


In [18]:
own[~own.type_result.str.startswith("correct")][["volume", "gold_id", "gold_title", "article_type", "type_result"]]

,volume,gold_id,gold_title,article_type,type_result


## 8. Issue metadata — number and date

In [19]:
issue = own.groupby(["volume", "issue_no"]).agg(
    pipeline_issue_no=("pipeline_issue_no", lambda x: ", ".join(sorted(set(map(str, x))))),
    pipeline_issue_date=("pipeline_issue_date", lambda x: ", ".join(sorted(set(map(str, x))))),
    number_correct=("issue_no_result", lambda x: (x == "correct").mean().round(2)),
    date_correct=("issue_date_result", lambda x: (x == "correct").mean().round(2)))
issue

pipeline_issue_no pipeline_issue_date  number_correct  date_correct
volume        issue_no                                                                    
chambers_1837 272                    272          04-15-1837             1.0           1.0
              273                    273          04-22-1837             1.0           1.0
              274                    274          04-29-1837             1.0           1.0
penny_1843    691                      8               01-07             0.0           1.0
              692                      9               01-14             0.0           1.0
              693                      9               01-14             0.0           0.0
saturday_1843 736                    736          12-23-1843             1.0           1.0
              737                    737          12-30-1843             1.0           1.0
              738                    738             12-1843             1.0           1.0

## 9. Imprints (publishing history)

In [20]:
if "imprints" in tables:
    display(tables["imprints"])
imprints[["volume", "gold_id", "issue_no", "result", "missing_part_found_in", "pipeline_imprint"]]

result,complete,partial,total
volume,,,
chambers_1837,3,0,3
saturday_1843,2,1,3
total,5,1,6


,volume,gold_id,issue_no,result,missing_part_found_in,pipeline_imprint
0,saturday_1843,34,736,complete,,"JOHN W. PARKER, PUBLISHER, WEST STRAND, LONDON"
1,saturday_1843,46,737,complete,,"JOHN W. PARKER, PUBLISHER, WEST STRAND, LONDON."
2,saturday_1843,49,738,partial,not found,"LONDON: JOHN W. PARKER, PUBLISHER, WEST STRAND"
3,chambers_1837,72,272,complete,,"EDINBURGH: Printed and published by W. and R. CHAMBERS, 19, Waterl..."
4,chambers_1837,104,273,complete,,"EDINBURGH: Printed and published by W. and R. CHAMBERS, 19, Waterl..."
5,chambers_1837,124,274,complete,,"EDINBURGH: Printed and published by W. and R. CHAMBERS, 19, Waterl..."


## 10. Errors beside Owen's item notes

Every item that is not exact, or that opens its own article with a field wrong, with the item note from the gold list.


In [21]:
notes = gold.set_index(["volume", "gold_id"])["item_notes"]
fields = ["title_result", "series_result", "figures_result"]
wrong = items[(items.status != "exact") | (items.opens_own_article & (
    ~items.title_result.isin(["correct", "both untitled"]) | ~items.series_result.isin(["correct", "none (both)"])
    | (items.figures_result == "wrong")))].copy()
wrong["item_notes"] = [notes.get((v, i), "") for v, i in zip(wrong.volume, wrong.gold_id)]
with pd.option_context("display.max_colwidth", 200):
    display(wrong[["volume", "gold_id", "start_pdf_page", "status", "gold_title", "pipeline_title", *fields, "item_notes"]])


,volume,gold_id,start_pdf_page,status,gold_title,pipeline_title,title_result,series_result,figures_result,item_notes
8,penny_1843,9,14,exact,The Castes and Tribes of India.,,missing,none (both),correct,"title and article under figure with caption, line rule below article"
9,penny_1843,10,15,merged,Walhalla,THE WALHALLA,correct,none (both),correct,"two footnotes with asterisk (also in text) below two different columns but no line rule separation for footnotes, only for the following article"
10,penny_1843,11,16,merged,Pekin.,,NaN,NaN,NaN,"Title in first line italics with dash, line rule above article, sign off end of text with dash, line rule below article end of column"
11,penny_1843,12,17,merged,Progresses of Queen Elizabeth. From Hatfield to her Coronation.,PROGRESSES OF QUEEN ELIZAВЕТН,wrong,none (both),correct,"Title and subtitle but no numbering below image with caption, possible series but calling it not. Asterisk lines * * * * * break up article into sections but with no section headers, indented vers..."
12,penny_1843,13,20,merged,Insurance of Shipping.—“Lloyd’s List.”,,NaN,NaN,NaN,"Title in first line italics with dash, line rule above article, sign off end of text with dash and italics"
15,penny_1843,16,23,exact,The Felling and transport of the Pine and Fire.,THE FELLING AND TRANSPORT OF THE PINE AND FIR,correct,none (both),wrong,"line rule above article, in-text reference to early article on lumbering in issue No. 352"
16,penny_1843,17,25,exact,Etruscan Antiquities at the British Museum.,ETRUSCAN ANTIQUITIES AT THE BRITISH MUSEUM,correct,none (both),wrong,"on p25 figure cuts page in half, dividing text columns horizontally, figure appears in right column on p 26"
20,saturday_1843,21,6,exact,Egyptian Sculpture.,AVIR EGYPTIAN SCULPTURE. HT,correct,none (both),wrong,"seemingly typical masthead with date alongside crest, “No: N” for issue number; first page features three images with captions, can be understood as one full page images with three parts, large “A..."
21,saturday_1843,22,7,lost,,,NaN,NaN,NaN,"short quote article with in text dash sign off, no title, line rule below article"
22,saturday_1843,23,7,exact,,COLOSSAL STATUE DRAWN TO ITS PLACE,extra (gold untitled),none (both),correct,"short quote article with in text dash sign off, no title, line rule below article"


## 11. Look up one gold item

In [22]:
def show(volume, gold_id):
    """A gold row next to what the pipeline produced for it."""
    row = gold[(gold.volume == volume) & (gold.gold_id == gold_id)].iloc[0]
    it = items[(items.volume == volume) & (items.gold_id == gold_id)]
    print("GOLD:", {k: v for k, v in row.items() if v and k not in ("pipeline_volume", "notes")})
    if len(it):
        it = it.iloc[0]
        print("\nRESULT:", it.status, "| title:", it.get("title_result", ""), "| merged with:", it.merged_with or "-")
        pipe = load_pipeline(VERSION, VOLUME_ALIASES.get(volume, volume))["articles"]
        ids = str(it.get("pipeline_articles") or it.start_article_id or "").split(";")
        for aid in [a for a in ids if a in pipe.index]:
            a = pipe.loc[aid]
            print(f"\n{aid}: {a.title!r}  pages {a.pages}")
            print("  front:", a.front_matter if isinstance(a.front_matter, str) else "-")
            print("  text :", " ".join(str(a.text).split()[:30]), "…")
            print("  back :", a.back_matter if isinstance(a.back_matter, str) else "-")

show("chambers_1837", 66)  # Hydrostatic Bed: its opening words are also quoted in the article before

GOLD: {'gold_id': 66, 'volume': 'chambers_1837', 'issue_no': '272', 'issue_date': '4.15.1837', 'start_pdf_page': 15, 'start_printed_page': '96', 'start_column': 'middle', 'end_pdf_page': '15', 'article_type': 'article', 'title': 'Hydrostatic Bed.', 'first_words': 'It is well known', 'last_words': 'like an ordinary bed.', 'back_matter': 'Newspaper paragraph.', 'item_notes': 'article with ending text sign off with a dash and italics, no line rule under article', 'end_page': 15}

RESULT: exact | title: correct | merged with: -

chambers_1837_0020: 'HYDROSTATIC BED'  pages 15
  front: -
  text : It is well known that the support of water to a floating body is so uniformly the inferior surface has, as it were, its own separate liquid pillar, diffused, that …
  back : -
